In [ ]:
# importing the required libraries
import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (Imputer,StringIndexer,OneHotEncoder,VectorAssembler,StandardScaler)
from pyspark.storagelevel import StorageLevel
import pandas as pd
import shutil
from google.colab import files

In [ ]:
# extracting the  combined parquet file for  all the month data from task1
ZIP_PATH = "/content/combined_yellow_taxi_2024.zip"
EXTRACT_PATH = "/content/extracted_combined"
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"ZIP file not found: {ZIP_PATH}")
if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The uploaded file is not a valid zip")
os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print("dataset extracted successfully")

dataset extracted successfully


In [ ]:
# creating session
spark = (SparkSession.builder.appName("NYC_Taxi_Task2").getOrCreate())

In [ ]:
# loading the combined data
DATA_PATH = os.path.join(EXTRACT_PATH,"combined_yellow_taxi_2024")
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(f"Dataset folder not found: {DATA_PATH}")
df = spark.read.parquet(DATA_PATH)
print("Dataset loaded successfully")

Dataset loaded successfully


In [ ]:
# no. of rows and columns
print("Total rows:", f"{df.count():,}")
print("Total columns:", len(df.columns))

In [ ]:
# schema of dataset
df.printSchema()

In [ ]:
df.show(15, truncate=False)

## Data Inspection

In [ ]:
# distribution of target column
df.groupBy("payment_type").count().orderBy("payment_type").show()

In [ ]:
# checking missing value and percentage
total_rows = df.count()
missing_data = df.select([F.sum(F.when(F.col(c).isNull() |(F.isnan(F.col(c)) if isinstance(df.schema[c].dataType, NumericType)else F.lit(False)),1).otherwise(0)).alias(c)for c in df.columns])
missing_data.show(truncate=False)
missing_percentage = missing_data.select([(F.col(c) / total_rows * 100).alias(c)for c in df.columns])
missing_percentage.show(truncate=False)

In [ ]:
# checking the value that are pesent in missing variables
columns_to_check = ["passenger_count","RatecodeID","store_and_fwd_flag","congestion_surcharge","Airport_fee"]
for col_name in columns_to_check:
    print(f"\nUnique values in {col_name}:")
    df.select(col_name).distinct().orderBy(col_name).show(100, truncate=False)

In [ ]:
# checking the value count of all the value present in each missing column
for col_name in columns_to_check:
    print(f"\nValue counts for {col_name}:")
    df.groupBy(col_name).count().orderBy(col_name).show(100, truncate=False)

In [ ]:
# partition count
partition_count = df.rdd.getNumPartitions()
print("number of Spark partitions:", partition_count)

In [ ]:
# checking the datatype for each column
for column_name, data_type in df.dtypes:
    print(f"{column_name}: {data_type}")

In [ ]:
# checking duplicate rows
unique_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - unique_rows
print("total rows:", total_rows)
print("unique rows:", unique_rows)
print("duplicate rows:", duplicate_rows)

In [ ]:
# dropping the duplicates
df = df.dropDuplicates()

In [ ]:
# rows left after the dropping duplicates
after_duplicates = df.count()
print("after duplicates removed:",after_duplicates)

In [ ]:
# count rows with missing pickup or drop-off timestamps
missing_timestamp_count = df.filter(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull()).count()

In [ ]:
# identify rows outside 2024 where both timestamps are present
outside_2024 = df.filter(F.col("tpep_pickup_datetime").isNotNull() & F.col("tpep_dropoff_datetime").isNotNull()
    & ((F.year("tpep_pickup_datetime") != 2024) | (F.year("tpep_dropoff_datetime") != 2024)))

In [ ]:
# Calculate the count rows outside the year
outside_2024_count = outside_2024.count()
outside_2024.select("tpep_pickup_datetime","tpep_dropoff_datetime").show(50, truncate=False)
print("Rows with missing timestamps:", missing_timestamp_count)
print("Records outside 2024:", outside_2024_count)

In [ ]:
# keep only rows with valid timestamps within 2024
df = df.filter(
    F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & (F.year("tpep_pickup_datetime") == 2024)
    & (F.year("tpep_dropoff_datetime") == 2024)
)
print("Rows remaining after year filtering:", df.count())

### Feature Engineering and Sampling

In [ ]:
# create a 10% random sample across all available months
sample_df = df.sample(withReplacement=False,fraction=0.10,seed=42).persist(StorageLevel.MEMORY_AND_DISK)
full_count = df.count()
sample_count = sample_df.count()
print(f"Full cleaned dataset: {full_count:,} rows")
print(f"Random sample: {sample_count:,} rows")
print(f"Actual sample percentage: {sample_count / full_count * 100:.2f}%")
'''
A 10% random sample was used for model development to reduce
computational cost. Its class distribution should be compared
with the full cleaned dataset to assess representativeness.
'''

In [ ]:
# Check class distribution
# payment_type values: 0 = Flex Fare trip 1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute, 5 = Unknown
print("class distribution in sampled dataset:") # Check class distribution after random sampling
sample_df.groupBy("payment_type").count().orderBy("payment_type").show()

In [ ]:
sample_df.show(10,truncate = False)

In [ ]:
# feature 1 -Identify longer distance trips, trip is labelled long when its distance exceeds 10 miles.
sample_df = sample_df.withColumn("is_long_trip",F.when(F.col("trip_distance") > 10, 1).otherwise(0))

In [ ]:
# feature 2 - # Feature 2: Extract the pickup hour (0–23) to identify hourly travel patterns.
sample_df = sample_df.withColumn("pickup_hour",F.hour("tpep_pickup_datetime"))

In [ ]:
# feature 3 -pickup day of the week , identifies the day when the trip started (1 = Sunday, 7 = Saturday).
sample_df = sample_df.withColumn("pickup_day_of_week",F.dayofweek("tpep_pickup_datetime"))

In [ ]:
# feature 4- weekend indicator - assigns 1 to Saturday/Sunday and 0 to weekdays.
sample_df = sample_df.withColumn("is_weekend",F.when(F.col("pickup_day_of_week").isin(1, 7), 1).otherwise(0))

In [ ]:
# feature 5-  pickup month - extracts the month of the pickup date (1–12).
sample_df = sample_df.withColumn("pickup_month",F.month("tpep_pickup_datetime"))

In [ ]:
# display the original columns and feature engineering columns
sample_df.show(10, truncate=False)


In [ ]:
# verify the updated dataset
print("Rows in sampled dataset:", f"{sample_df.count():,}")
print("Original columns:", len(df.columns))
print("Columns after feature engineering:", len(sample_df.columns))

In [ ]:
for column_name, data_type in sample_df.dtypes:
    print(f"{column_name}: {data_type}")

In [ ]:
# Export sample_df as CSV
csv_path = "/content/sample"
sample_df.coalesce(1).write.mode("overwrite").option("header", True).option("encoding", "UTF-8").csv(csv_path)
print("10% sample exported successfully:", csv_path)

### Data preprocessing

In [ ]:
# selecting appropriate model predictors
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend",
    "pickup_month",
    "is_long_trip"]

categorical_cols = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID"]

target_col = "payment_type"
'''Leakage prevention:
Fare-related and payment-related variables were excluded:
fare_amount,tip_amount,total_amount,extra,tolls_amount
These variables are generated after or during payment processing
and could reveal the target payment_type.'''

In [ ]:
# check that every required column exists
required_cols = numeric_cols + categorical_cols + [target_col]
missing_cols = [
    col_name for col_name in required_cols
    if col_name not in sample_df.columns]
if missing_cols:
    raise ValueError(f"These required columns are missing: {missing_cols}")

In [ ]:
# keep only the columns used by this modeling pipeline
model_df = sample_df.select(*required_cols)
print("columns selected for modelling:")
print(model_df.columns)
print("number of rows before preprocessing:", model_df.count())

In [ ]:
# clean numerical predictor columns
for col_name in numeric_cols:
    model_df = model_df.withColumn(
        col_name,
        F.col(col_name).cast("double"))

In [ ]:
# converting invalid infinite values and NaNs to null
for col_name in numeric_cols:
    model_df = model_df.withColumn(col_name,F.when(F.col(col_name).isNull() | F.isnan(F.col(col_name)) | (F.abs(F.col(col_name)) == float("inf")),None).otherwise(F.col(col_name)))

In [ ]:
# Convert category values to strings and represent missing categorical values as "Unknown".
for col_name in categorical_cols:
    model_df = model_df.withColumn(
        col_name,F.coalesce(F.col(col_name).cast("string"),F.lit("Unknown")))

In [ ]:
# Prepare the classification target
model_df = (model_df.withColumn(target_col, F.col(target_col).cast("int")).filter(F.col(target_col).isNotNull()).filter(F.col(target_col).between(0, 6)))
print("Rows after target cleaning:", model_df.count())
print("Payment type distribution:")
model_df.groupBy(target_col).count().orderBy(target_col).show()

In [ ]:
# split into training and testing datasets
train_df, test_df = model_df.randomSplit([0.70, 0.30],seed=42)
print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [ ]:
# verify that every test target class also occurs in training.
train_classes = {row[target_col] for row in train_df.select(target_col).distinct().collect()}
test_classes = {row[target_col] for row in test_df.select(target_col).distinct().collect()}

In [ ]:
# verify that every payment_type class present in the testing dataset
missing_train_classes = test_classes - train_classes
if missing_train_classes:
    raise ValueError(f"Classes present in test but absent from training: "
        f"{missing_train_classes}")
print("All test classes are present in training.")

In [ ]:
# configure numerical imputation
imputed_cols = [f"{col_name}_imputed" for col_name in numeric_cols]
imputer = Imputer(inputCols=numeric_cols, outputCols=imputed_cols,strategy="median")

In [ ]:
# encode categorical predictors
indexed_cols = [f"{col_name}_indexed" for col_name in categorical_cols]
encoded_cols = [f"{col_name}_encoded" for col_name in categorical_cols]
category_indexer = StringIndexer(inputCols=categorical_cols,outputCols=indexed_cols,handleInvalid="keep")
one_hot_encoder = OneHotEncoder(inputCols=indexed_cols,outputCols=encoded_cols,handleInvalid="keep",dropLast=False)

In [ ]:
# assemble for feature transformation and scale predictor features
assembler = VectorAssembler(inputCols=imputed_cols + encoded_cols,outputCol="unscaled_features",handleInvalid="error")
scaler = StandardScaler(inputCol="unscaled_features",outputCol="features",withStd=True,withMean=False)

In [ ]:
# convert payment_type into a classification label
label_indexer = StringIndexer(inputCol=target_col,outputCol="label",handleInvalid="error")

In [ ]:
# build and fit the complete preprocessing pipeline
preprocessing_pipeline = Pipeline(stages=[imputer,category_indexer,one_hot_encoder,assembler,scaler,label_indexer])
preprocessing_model = preprocessing_pipeline.fit(train_df) # fit using training data only
train_transformed = preprocessing_model.transform(train_df) # apply the fitted transformations to both datasets
test_transformed = preprocessing_model.transform(test_df)

In [ ]:
# display encoded labels and their original payment_type values
for stage in preprocessing_model.stages:
    if hasattr(stage, "labelsArray") and stage.getOutputCol() == "label":
        label_mapping = {
            float(index): value
            for index, value in enumerate(stage.labelsArray[0])
        }
        print("Encoded label -> Original payment_type")
        for encoded_label, original_value in label_mapping.items():
            print(f"{encoded_label} -> {original_value}")

In [ ]:
# create datasets for modeling
train_prepared = train_transformed.select("features", "label")
test_prepared = test_transformed.select("features", "label")

In [ ]:
print("Training data prepared:")
train_prepared.show(5, truncate=False)
print("Testing data prepared for modeling:")
test_prepared.show(5, truncate=False)

In [ ]:
# verify prepared datasets
print("Training rows:", train_prepared.count())
print("Testing rows:", test_prepared.count())
print("Training prepared data:")
train_prepared.show(5, truncate=False)
print("Testing prepared data:")
test_prepared.show(5, truncate=False)

In [ ]:
# checking for missing features or labels
train_null_count = train_prepared.filter(F.col("features").isNull() | F.col("label").isNull()).count()
test_null_count = test_prepared.filter(F.col("features").isNull() | F.col("label").isNull()).count()
assert train_null_count == 0, "Training data contains null features or labels"
assert test_null_count == 0, "Testing data contains null features or labels"

In [ ]:
# verify feature vector sizes match
train_feature_size = (train_prepared.select("features").first()["features"].size)
test_feature_size = (test_prepared.select("features").first()["features"].size)
print("Training feature vector size:", train_feature_size)
print("Testing feature vector size:", test_feature_size)
assert train_feature_size == test_feature_size, ("Training and testing feature vector sizes do not match.")
print("Pipeline transformation checks passed")

In [ ]:
# save the fitted preprocessing pipeline
PIPELINE_PATH = "/content/nyc_taxi_preprocessing_pipeline"
preprocessing_model.write().overwrite().save(PIPELINE_PATH)
print("preprocessing pipeline saved to:", PIPELINE_PATH)

In [ ]:
# define output paths for the prepared training and testing datasets.
TRAIN_PATH = "/content/train_prepared"
TEST_PATH = "/content/test_prepared"
train_prepared.write.mode("overwrite").parquet(TRAIN_PATH)
test_prepared.write.mode("overwrite").parquet(TEST_PATH)
print("Training data exported to:", TRAIN_PATH)
print("Testing data exported to:", TEST_PATH)

In [ ]:
# Check that both folders exist
TRAIN_PATH = "/content/train_prepared"
TEST_PATH = "/content/test_prepared"
print("Training folder exists:", os.path.exists(TRAIN_PATH))
print("Testing folder exists:", os.path.exists(TEST_PATH))
shutil.make_archive("/content/train_prepared","zip",root_dir=TRAIN_PATH)
shutil.make_archive("/content/test_prepared","zip",root_dir=TEST_PATH)
print("Both folders have been compressed successfully.")

In [ ]:
files.download("/content/train_prepared.zip")
files.download("/content/test_prepared.zip")